# MNIST MLP3 — AdamW baseline

This is the clean AdamW control: 55,000 optimization examples, a fixed 5,000-example validation split, and the official test set used only for monitoring. AdamW uses step-level warm-up/cosine decay from 1e-3 to 1e-5, betas (0.90, 0.999), matrix-only weight decay 1e-2, and gradient clipping at 1.0.

At epoch zero and every epoch the code runs `watcher.analyze(ERG=True, randomize=True)` and retains direct `alpha`, `ERG_gap`, and `num_traps`. The standard plot bundle includes `7_layerwise_weightwatcher_num_traps_95ci.png`.

In [ ]:
from pathlib import Path
import os
import sys
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = None
for path in [Path.cwd(), *Path.cwd().parents]:
    candidate = path / 'baseline'
    if (candidate / 'rg_baselines').is_dir(): ROOT = candidate; break
    if (path / 'rg_baselines').is_dir(): ROOT = path; break
if ROOT is None: raise RuntimeError('Run from a clone of CalculatedContent/rg_optimizers.')
ROOT = ROOT.resolve()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from rg_baselines import BaselineConfig, DEFAULT_BASELINE_SEEDS, plot_all_replicates, run_baseline_replicates
RUN_ROOT = Path(os.environ.get('RG_BASELINE_RUN_ROOT', ROOT / 'runs')).expanduser().resolve()
DATA_DIR = Path(os.environ.get('RG_BASELINE_DATA_DIR', ROOT / 'data')).expanduser().resolve()
RUN_DIR = RUN_ROOT / 'adamw'
PLOT_DIR = RUN_DIR / 'plots'
for directory in (RUN_ROOT, DATA_DIR, RUN_DIR, PLOT_DIR): directory.mkdir(parents=True, exist_ok=True)

In [ ]:
CONFIG = BaselineConfig(
    optimizer='adamw', epochs=30, validation_size=5_000,
    adamw_learning_rate=1e-3, adamw_min_learning_rate=1e-5,
    adamw_warmup_epochs=1, adamw_beta1=0.90, adamw_beta2=0.999,
    adamw_weight_decay=1e-2, ww_randomize=True, save_epoch_checkpoints=True,
)
SEEDS = DEFAULT_BASELINE_SEEDS
assert len(SEEDS) == 3 and len(set(SEEDS)) == 3
display(pd.DataFrame([CONFIG.__dict__]))

## Run or resume three complete trajectories

Latest, validation-selected, final, and per-epoch checkpoints are persisted. Complete compatible runs are loaded; incomplete runs resume with all optimizer and RNG state.

In [ ]:
suite = run_baseline_replicates(
    CONFIG, seeds=SEEDS, data_dir=DATA_DIR, output_dir=RUN_DIR,
    progress=True, confidence=0.95, resume=True,
)
plot_all_replicates(suite, output_dir=PLOT_DIR, show=True)

## Train, validation, test, and learning-rate trajectories

All bands are two-sided 95% Student-t intervals across complete runs. Validation loss is the only checkpoint-selection signal.

In [ ]:
for metric in ['train_loss','validation_loss','test_loss','train_accuracy','validation_accuracy','test_accuracy','primary_lr']:
    summary = suite.performance_summary[suite.performance_summary['metric'].eq(metric)].sort_values('epoch')
    figure, axis = plt.subplots(figsize=(9, 5))
    for seed, run in suite.performance.groupby('seed'): axis.plot(run['epoch'], run[metric], alpha=0.18, linewidth=0.8)
    axis.plot(summary['epoch'], summary['mean'], linewidth=2.0, label='mean')
    axis.fill_between(summary['epoch'], summary['ci_low'], summary['ci_high'], alpha=0.16)
    axis.set(xlabel='Epoch', ylabel=metric.replace('_',' ').title(), title=f'{CONFIG.optimizer_label}: {metric}')
    axis.grid(alpha=0.25); axis.legend(frameon=False); figure.tight_layout()
    figure.savefig(PLOT_DIR / f'{metric}_95ci.png', dpi=170, bbox_inches='tight')
    plt.show()
display(suite.performance_summary[suite.performance_summary['metric'].isin(['train_loss','validation_loss','test_loss','train_accuracy','validation_accuracy','test_accuracy','primary_lr'])].sort_values(['metric','epoch']))

## Layerwise WeightWatcher diagnostics

In [ ]:
required = ['alpha','ERG_gap','num_traps','m_midpoint','trace_log_midpoint_per_eval']
rows = suite.spectral_summary[suite.spectral_summary['metric'].isin(required)].sort_values(['metric','layer','epoch'])
assert rows['n'].eq(3).all()
display(rows)

In [ ]:
required_paths = [RUN_DIR / 'performance_by_epoch_and_seed.csv', RUN_DIR / 'spectral_metrics_by_epoch_layer_and_seed.csv', RUN_DIR / 'performance_summary_95ci.csv', RUN_DIR / 'spectral_summary_95ci.csv', RUN_DIR / 'replicate_manifest.json', PLOT_DIR / '7_layerwise_weightwatcher_num_traps_95ci.png']
for seed in SEEDS:
    seed_dir = RUN_DIR / 'seeds' / f'seed_{seed}'
    required_paths.extend([seed_dir / 'checkpoint_latest.pt', seed_dir / 'checkpoint_best.pt', seed_dir / 'final_state.pt', seed_dir / 'test_results.json', seed_dir / 'run_complete.json'])
missing = [path for path in required_paths if not path.is_file()]
if missing: raise RuntimeError('Missing required artifacts:\n' + '\n'.join(map(str, missing)))
print('verified artifacts:', len(required_paths))